# 实践 4.1  群体结构如何制造GWAS假阳性

《统计遗传与智能育种学》第四章 · 电子资源 `04_gwas_population_structure.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：04_gwas_population_structure.ipynb。实践模拟两个亚群的基因型和数量性状，在已知因果位点之外人为加入亚群表型差异。分别运行朴素单标记扫描和加入基因型主成分（principal component, PC）的校正扫描，比较P值分布、非因果位点假阳性数量以及真实因果位点的排序。该实践用可解释的小模型演示混杂，不把简化实现替代全基因组高效混合模型关联分析（genome-wide efficient mixed-model association, GEMMA）、GCTA等正式分析软件。

核心比较：未校正GWAS vs 主成分分析（principal component analysis, PCA）校正GWAS  
验收：固定随机种子；输出P值摘要、非因果假阳性数和真实因果位点排名；结果可重复。

## 输入、完整代码、实际输出与解释

输入与可复现条件：两个具有等位基因频率差异的模拟亚群、1个真实因果SNP并人为加入亚群表型差异；SEED=401；无需外部文件。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED=401
rng=np.random.default_rng(SEED)
print('Python',sys.version.split()[0])
print('Platform',platform.platform())
print('numpy',np.__version__,'pandas',pd.__version__,'scikit-learn',sklearn.__version__)
print('SEED',SEED)


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 401


In [2]:
from scipy.stats import t as tdist
from sklearn.decomposition import PCA

n,m=420,700; grp=np.r_[np.zeros(n//2),np.ones(n//2)]; p0=rng.uniform(.08,.45,m); p1=np.clip(p0+rng.normal(.10,.07,m),.03,.95); X=np.vstack([rng.binomial(2,p0,(n//2,m)),rng.binomial(2,p1,(n//2,m))]).astype(float)
causal=123; y=.9*X[:,causal]+1.8*grp+rng.normal(0,1,n)
PC=PCA(n_components=3,random_state=SEED).fit_transform((X-X.mean(0))/np.maximum(X.std(0),1e-6))
def scan(covar=None):
 out=[]
 C=np.ones((n,1)) if covar is None else np.column_stack([np.ones(n),covar])
 for j in range(m):
  D=np.column_stack([C,X[:,j]]); bh=np.linalg.lstsq(D,y,rcond=None)[0]; r=y-D@bh; df=n-D.shape[1]; s2=(r@r)/df; se=np.sqrt(s2*np.linalg.inv(D.T@D)[-1,-1]); tv=bh[-1]/se; out.append(2*tdist.sf(abs(tv),df))
 return np.array(out)
p_naive=scan(); p_pc=scan(PC[:,:2]); thr=.001; non=np.arange(m)!=causal
print('P-value median naive/PC:',round(float(np.median(p_naive)),4),round(float(np.median(p_pc)),4))
print('P-value 1% quantile naive/PC:',round(float(np.quantile(p_naive,.01)),6),round(float(np.quantile(p_pc,.01)),6))
print('false positives naive/PC:',int(np.sum((p_naive<thr)&non)),int(np.sum((p_pc<thr)&non)))
print('causal rank naive/PC:',int(np.argsort(p_naive).tolist().index(causal)+1),int(np.argsort(p_pc).tolist().index(causal)+1))


P-value median naive/PC: 0.0328 0.5034
P-value 1% quantile naive/PC: 0.0 0.014868
false positives naive/PC: 158 1
causal rank naive/PC: 1 1


结果解释：在单标记阈值P<0.001下，朴素扫描的P值中位数降至0.0328并出现158个非因果假阳性，加入基因型PC后中位数约0.5034、假阳性降至1；真实因果位点在两种扫描中均排第1。本例说明群体结构可制造系统性关联，但PC校正并不是所有亲缘结构的通用替代，实际GWAS分析常需混合模型。

可修改练习：逐步把亚群表型差异设为0、1、2，比较Q-Q偏离和假阳性数；再改变PC个数，观察过少或过多协变量的影响。